# PrimeKG Dataset Exploratory Data Analysis

This notebook explores the local PrimeKG dataset used for the BioKG Explorer project. The goal is to understand the graph schema, node and relation distributions, feature tables, and practical constraints for building an interactive biomedical knowledge graph visualization tool.

Dataset location expected by this notebook: `../prime-kg-dataset/`.

## 1. Environment Setup

Import the core Python libraries used throughout this notebook and configure plotting defaults. The notebook intentionally uses common dependencies (`pandas`, `matplotlib`) so it can run in a lightweight local environment.

In [ ]:
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
plt.style.use("default")
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Dataset Paths

Define the dataset directory and the expected PrimeKG files. This cell also verifies that the files are present before the rest of the analysis runs.

In [ ]:
DATA_DIR = Path("../prime-kg-dataset").resolve()

expected_files = [
    "README.txt",
    "nodes.csv",
    "edges.csv",
    "kg.csv",
    "disease_features.csv",
    "drug_features.csv",
    "kg_raw.csv",
    "kg_giant.csv",
    "kg_grouped.csv",
    "kg_grouped_diseases.csv",
    "kg_grouped_diseases_bert_map.csv",
]

file_status = pd.DataFrame(
    {
        "file": expected_files,
        "exists": [(DATA_DIR / name).exists() for name in expected_files],
        "size_mb": [
            round((DATA_DIR / name).stat().st_size / (1024**2), 2) if (DATA_DIR / name).exists() else None
            for name in expected_files
        ],
    }
)

print(f"Dataset directory: {DATA_DIR}")
file_status

## 3. README Notes

Read the dataset README so the analysis remains grounded in the dataset author's description of each file.

In [ ]:
readme_text = (DATA_DIR / "README.txt").read_text(encoding="utf-8")
print(readme_text)

## 4. CSV Schemas and Row Counts

Inspect each CSV header and count rows. The edge-level files are large, so this uses Python's CSV reader instead of loading every file into memory with pandas.

In [ ]:
import csv

def csv_header_and_rows(path: Path):
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        row_count = sum(1 for _ in reader)
    return header, row_count

schema_rows = []
for name in expected_files:
    path = DATA_DIR / name
    if path.suffix == ".csv" and path.exists():
        header, row_count = csv_header_and_rows(path)
        schema_rows.append({"file": name, "rows": row_count, "columns": header})

schemas = pd.DataFrame(schema_rows).sort_values("file")
schemas

## 5. Load Core Node Table

Load `nodes.csv`, which is small enough for memory. This table is the canonical node lookup keyed by `node_index`.

In [ ]:
nodes = pd.read_csv(DATA_DIR / "nodes.csv")

print(nodes.shape)
nodes.head()

## 6. Node Quality Checks

Check whether key node fields are unique and complete. These checks help decide whether `node_index`, `node_id`, or a composite key should be used in a backend database.

In [ ]:
node_quality = pd.Series(
    {
        "rows": len(nodes),
        "unique_node_index": nodes["node_index"].nunique(),
        "unique_node_id": nodes["node_id"].nunique(),
        "duplicate_node_index_rows": nodes.duplicated("node_index").sum(),
        "duplicate_node_id_rows": nodes.duplicated("node_id").sum(),
        "missing_node_names": nodes["node_name"].isna().sum(),
        "missing_node_sources": nodes["node_source"].isna().sum(),
    }
)

node_quality.to_frame("value")

## 7. Node Type Distribution

Summarize and visualize node counts by biomedical entity type. This is one of the most important views for designing graph filters and visual encodings.

In [ ]:
node_type_counts = nodes["node_type"].value_counts().rename_axis("node_type").reset_index(name="count")
node_type_counts["pct"] = (node_type_counts["count"] / node_type_counts["count"].sum() * 100).round(2)

ax = node_type_counts.plot.bar(x="node_type", y="count", legend=False)
ax.set_title("PrimeKG Node Counts by Type")
ax.set_xlabel("Node type")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

node_type_counts

## 8. Node Source Distribution

Summarize the source ontology or database for each node. This is useful for provenance labels and source-specific filters in the visualization interface.

In [ ]:
node_source_counts = nodes["node_source"].value_counts().rename_axis("node_source").reset_index(name="count")
node_source_counts["pct"] = (node_source_counts["count"] / node_source_counts["count"].sum() * 100).round(2)

ax = node_source_counts.plot.bar(x="node_source", y="count", legend=False)
ax.set_title("PrimeKG Node Counts by Source")
ax.set_xlabel("Node source")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

node_source_counts

## 9. Stream Edge Relation Counts

`edges.csv` has more than 8 million rows, so this cell processes it in chunks. It counts relation types and display relation labels without loading the full edge list into memory.

In [ ]:
EDGE_CHUNKSIZE = 500_000

relation_counter = Counter()
display_relation_counter = Counter()
edge_rows = 0

for chunk in pd.read_csv(DATA_DIR / "edges.csv", chunksize=EDGE_CHUNKSIZE):
    edge_rows += len(chunk)
    relation_counter.update(chunk["relation"].dropna())
    display_relation_counter.update(chunk["display_relation"].dropna())

relation_counts = pd.DataFrame(relation_counter.most_common(), columns=["relation", "count"])
relation_counts["pct"] = (relation_counts["count"] / edge_rows * 100).round(2)

display_relation_counts = pd.DataFrame(display_relation_counter.most_common(), columns=["display_relation", "count"])
display_relation_counts["pct"] = (display_relation_counts["count"] / edge_rows * 100).round(2)

print(f"Total edge rows: {edge_rows:,}")
relation_counts.head(20)

## 10. Relation Distribution Plot

Visualize the most common relation types. The distribution is highly skewed, which has implications for default filters and backend query limits.

In [ ]:
top_n = 20
ax = relation_counts.head(top_n).sort_values("count").plot.barh(x="relation", y="count", legend=False)
ax.set_title(f"Top {top_n} PrimeKG Relation Types")
ax.set_xlabel("Edge count")
ax.set_ylabel("Relation")
plt.tight_layout()

## 11. Display Relation Labels

Inspect the human-readable relation labels. These labels are better candidates for UI text than raw relation identifiers.

In [ ]:
display_relation_counts

## 12. Edge Endpoint Type Pairs

Join edge endpoints to node types in a streaming pass. This reveals which node-type combinations are connected by each relation and helps design allowed filter combinations.

In [ ]:
node_index_to_type = nodes.set_index("node_index")["node_type"].to_dict()

type_pair_counter = Counter()
missing_endpoint_rows = 0

for chunk in pd.read_csv(DATA_DIR / "edges.csv", chunksize=EDGE_CHUNKSIZE):
    x_type = chunk["x_index"].map(node_index_to_type)
    y_type = chunk["y_index"].map(node_index_to_type)
    missing_endpoint_rows += (x_type.isna() | y_type.isna()).sum()
    type_pair_counter.update(zip(chunk["relation"], x_type, y_type))

type_pair_counts = pd.DataFrame(
    [(relation, x_type, y_type, count) for (relation, x_type, y_type), count in type_pair_counter.items()],
    columns=["relation", "x_type", "y_type", "count"],
).sort_values("count", ascending=False)

print(f"Rows with missing endpoint type: {missing_endpoint_rows:,}")
type_pair_counts.head(30)

## 13. Check Edge Symmetry

The README describes `edges.csv` as undirected. This cell checks how many unordered endpoint pairs have both directions represented. It is useful for deciding whether the app should collapse reciprocal rows for display.

In [ ]:
edge_pairs = set()
reciprocal_rows = 0
sample_limit = None  # Set to an integer such as 1_000_000 for a faster approximate check.
rows_seen = 0

for chunk in pd.read_csv(DATA_DIR / "edges.csv", usecols=["relation", "x_index", "y_index"], chunksize=EDGE_CHUNKSIZE):
    for relation, x_index, y_index in chunk.itertuples(index=False):
        rows_seen += 1
        if (relation, y_index, x_index) in edge_pairs:
            reciprocal_rows += 1
        edge_pairs.add((relation, x_index, y_index))
    if sample_limit and rows_seen >= sample_limit:
        break

symmetry_summary = pd.Series(
    {
        "rows_checked": rows_seen,
        "unique_directed_relation_pairs": len(edge_pairs),
        "rows_with_previously_seen_reverse": reciprocal_rows,
    }
)

symmetry_summary.to_frame("value")

## 14. Load Disease Features

Load disease-level text and clinical features. This file can power disease detail panels, search snippets, and text-based summaries.

In [ ]:
disease_features = pd.read_csv(DATA_DIR / "disease_features.csv")

print(disease_features.shape)
disease_features.head()

## 15. Disease Feature Completeness

Measure non-null coverage for each disease feature column. Sparse fields should be handled carefully in the UI, while high-coverage fields are good candidates for default display.

In [ ]:
disease_feature_coverage = pd.DataFrame(
    {
        "column": disease_features.columns,
        "non_null": disease_features.notna().sum().values,
        "non_null_pct": (disease_features.notna().mean().values * 100).round(2),
    }
).sort_values("non_null_pct", ascending=False)

disease_feature_coverage

## 16. Load Drug Features

Load drug-level descriptive and pharmacological attributes. These fields can support drug detail panels, faceted search, and tooltip content.

In [ ]:
drug_features = pd.read_csv(DATA_DIR / "drug_features.csv")

print(drug_features.shape)
drug_features.head()

## 17. Drug Feature Completeness

Measure non-null coverage for drug feature columns. This helps identify which drug attributes are consistently available enough for primary UI placement.

In [ ]:
drug_feature_coverage = pd.DataFrame(
    {
        "column": drug_features.columns,
        "non_null": drug_features.notna().sum().values,
        "non_null_pct": (drug_features.notna().mean().values * 100).round(2),
    }
).sort_values("non_null_pct", ascending=False)

drug_feature_coverage

## 18. Disease Grouping Files

Inspect grouped disease files. Disease grouping can reduce visual clutter by collapsing many fine-grained MONDO disease nodes into higher-level disease groups.

In [ ]:
grouped_diseases = pd.read_csv(DATA_DIR / "kg_grouped_diseases.csv")
bert_group_map = pd.read_csv(DATA_DIR / "kg_grouped_diseases_bert_map.csv")

print("kg_grouped_diseases:", grouped_diseases.shape)
print("kg_grouped_diseases_bert_map:", bert_group_map.shape)

grouped_diseases.head()

## 19. Disease Group Size Distribution

Summarize the size of BERT-derived disease groups. Large groups may be useful aggregate nodes; singleton or small groups may be better left as individual diseases.

In [ ]:
group_size_counts = (
    grouped_diseases.dropna(subset=["group_name_bert"])
    .groupby("group_name_bert")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="disease_count")
)

ax = group_size_counts.head(25).sort_values("disease_count").plot.barh(
    x="group_name_bert", y="disease_count", legend=False
)
ax.set_title("Largest Disease Groups by BERT Group Name")
ax.set_xlabel("Disease count")
ax.set_ylabel("Disease group")
plt.tight_layout()

group_size_counts.head(25)

## 20. Search Nodes by Name

Define a small helper for searching nodes by case-insensitive name substring. This is a useful starting point for choosing interesting seed nodes for graph exploration.

In [ ]:
def search_nodes(query, node_type=None, limit=20):
    mask = nodes["node_name"].str.contains(query, case=False, na=False)
    if node_type is not None:
        mask &= nodes["node_type"].eq(node_type)
    return nodes.loc[mask].head(limit)

search_nodes("diabetes")

## 21. Extract a One-Hop Neighborhood

Define a chunked helper that retrieves a one-hop neighborhood around a selected `node_index`. This mirrors the kind of backend query needed for an interactive graph viewer.

In [ ]:
def one_hop_edges(seed_node_index, relation_filter=None, max_edges=2_000):
    matches = []
    usecols = ["relation", "display_relation", "x_index", "y_index"]

    for chunk in pd.read_csv(DATA_DIR / "edges.csv", usecols=usecols, chunksize=EDGE_CHUNKSIZE):
        mask = chunk["x_index"].eq(seed_node_index) | chunk["y_index"].eq(seed_node_index)
        if relation_filter is not None:
            allowed = {relation_filter} if isinstance(relation_filter, str) else set(relation_filter)
            mask &= chunk["relation"].isin(allowed)

        found = chunk.loc[mask]
        if not found.empty:
            matches.append(found)

        if sum(len(frame) for frame in matches) >= max_edges:
            break

    if not matches:
        return pd.DataFrame(columns=usecols)

    result = pd.concat(matches, ignore_index=True).head(max_edges)
    return result

# Example: replace this with a node_index from search_nodes(...) above.
example_seed = int(search_nodes("diabetes", node_type="disease").iloc[0]["node_index"])
example_edges = one_hop_edges(example_seed, max_edges=100)
example_edges.head()

## 22. Enrich Neighborhood Edges with Node Metadata

Join one-hop edges back to node metadata so the neighborhood is readable. This output shape is close to what an API endpoint could return to a frontend visualization.

In [ ]:
def enrich_edges(edge_df):
    x_nodes = nodes.add_prefix("x_").rename(columns={"x_node_index": "x_index"})
    y_nodes = nodes.add_prefix("y_").rename(columns={"y_node_index": "y_index"})
    return edge_df.merge(x_nodes, on="x_index", how="left").merge(y_nodes, on="y_index", how="left")

enriched_example_edges = enrich_edges(example_edges)
enriched_example_edges.head(20)

## 23. Neighborhood Node Type Mix

Summarize the entity types found in a seed node's one-hop neighborhood. This can inform default colors, legends, and neighbor prioritization in the visualization.

In [ ]:
def neighborhood_node_type_mix(edge_df, seed_node_index):
    if edge_df.empty:
        return pd.DataFrame(columns=["node_type", "count"])

    neighbor_indices = pd.concat(
        [
            edge_df.loc[edge_df["x_index"].eq(seed_node_index), "y_index"],
            edge_df.loc[edge_df["y_index"].eq(seed_node_index), "x_index"],
        ]
    )
    neighbor_nodes = nodes[nodes["node_index"].isin(neighbor_indices)]
    return neighbor_nodes["node_type"].value_counts().rename_axis("node_type").reset_index(name="count")

neighborhood_node_type_mix(example_edges, example_seed)

## 24. Initial Visualization Design Notes

Use the observations from this EDA to capture early product and engineering implications for BioKG Explorer.

In [ ]:
design_notes = pd.DataFrame(
    [
        {
            "observation": "The graph has millions of edges and several highly dominant relation types.",
            "implication": "Do not render the full graph in the browser; use backend-filtered subgraphs and pagination/limits.",
        },
        {
            "observation": "Disease, drug, and gene/protein nodes have rich biomedical value for users.",
            "implication": "Start with disease-centered search and one-hop expansion across drugs, genes, phenotypes, and pathways.",
        },
        {
            "observation": "Feature tables contain long text fields with uneven coverage.",
            "implication": "Use feature panels with graceful handling for missing fields instead of dense graph labels.",
        },
        {
            "observation": "Disease grouping files are available.",
            "implication": "Offer grouped disease exploration or aggregate views to reduce visual clutter.",
        },
    ]
)

design_notes